# Memory and Conversation State

In notebook 01, the agent handled each request on its own. Real support conversations are not like that: a customer follows up about "this ticket", and useful facts about them should carry over to the next conversation. Memory makes this possible, but it is an explicit engineering choice: which facts to retain, how long to retain them, how to retrieve them, and what must never be stored.

In this notebook, you carry message history into a follow-up turn and filter stored customer notes before they reach the Northstar operations assistant.

## Learning Objectives

- Reuse message history across conversation turns.
- Distinguish conversation state from stored memory.
- Filter recalled notes by confidence and content without modifying storage.

## Mental Model

A language model has no memory of its own: it only knows what is in the current request. Everything an agent "remembers" is context the application puts back into that request, from two sources:

- **Short-term:** the earlier messages of this conversation, sent again with every request, like a support agent remembering what the caller just said.
- **Long-term:** stored notes about the customer, retrieved and filtered so only reliable, safe notes reach the model, like a customer file read between calls.


```mermaid
flowchart LR
    subgraph Session["Short#8209;term:&nbsp;one&nbsp;conversation"]
        P1["Turn 1: Plan T-1001"] --> H["message_history<br/>(all_messages)"]
        H --> P2["Turn 2: follow-up<br/>without ticket ID"]
    end
    subgraph Durable["Long-term: durable notes"]
        S["seed_memory.json"] --> R["recall_memory(customer)"]
        R --> F{"confidence >= 0.85<br/>and is_safe_memory?"}
        F -- yes --> K["Note reaches the model"]
        F -- no --> X["Note withheld"]
    end
    K --> C["Context for the next decision"]
    P2 --> C
```

Research on language agents names these memory types explicitly. In the CoALA framework, working memory holds what the current decision needs, while long-term memories are read through retrieval and updated through learning. In this notebook, message history plays the role of working context, and `seed_memory.json` is a small long-term store that only reaches the model through `recall_memory`.

<div align="center">
  <img src="assets/02-coala-agent-architecture.png" alt="CoALA language agent with working memory and long-term memories" width="700" style="max-width: 100%; height: auto;" />
  <p><em>A language agent in the CoALA framework: a decision procedure, working memory, and procedural, semantic, and episodic long-term memories, connected to the environment through actions and observations. Source: <a href="https://arxiv.org/abs/2309.02427" target="_blank" rel="noopener noreferrer">Sumers et al., 2023</a>, CC BY 4.0</em></p>
</div>

## Load The Shared Helpers


In [ ]:
from support import (
    MemoryNote,
    agent_limits,
    build_agent,
    is_safe_memory,
    load_operations_data,
    trace_from_result,
)

## Short-Term Memory: Message History

Short-term memory keeps the conversation coherent inside the current session. Pass the first run's `all_messages()` back through `message_history` when the next request omits the ticket ID. Both scripted and live models receive that history, not hidden notebook variables.


In [ ]:
agent = build_agent()
limits = agent_limits()
print("Model:", type(agent.model).__name__)
first_result = await agent.run("Plan the next action for T-1001.", usage_limits=limits)
message_history = first_result.all_messages()
follow_up = await agent.run(
    "Who should take the next action on this ticket?",
    message_history=message_history,
    usage_limits=limits,
)
follow_up.output.model_dump()

Now send the same follow-up without `message_history`. The scripted model stops with an error because no ticket is named. A live model should ask for the ticket ID; inspect its response for unsupported guesses.

In [ ]:
try:
    without_history = await agent.run(
        "Who should take the next action on this ticket?", usage_limits=limits
    )
    print(without_history.output.model_dump())
except ValueError as error:
    print(f"Without history: {error}")

The history is a list of model requests and responses. It holds the user prompt, every tool call the model proposed, and every tool result, so the next run sees the whole previous turn. Notice what the follow-up did with it: the history gave the model the ticket ID, not cached results, so it called the tools again. The last call, `final_result`, is the output tool PydanticAI uses to return the validated `AgentDecision`.

In [ ]:
for message in message_history:
    print(type(message).__name__, [type(part).__name__ for part in message.parts])

follow_up_calls = [
    part.tool_name
    for message in follow_up.new_messages()
    for part in message.parts
    if part.part_kind == "tool-call"
]
print("Follow-up tool calls:", follow_up_calls)

## Durable Memory

Message history only lasts for one session, and it grows with every turn: each new request resends all earlier messages, which adds cost and eventually hits the model's context limit. Durable memory solves a different problem. It keeps a few selected facts across sessions, and because those facts are added to future requests, it should be narrow. Store operationally useful preferences and context, not credentials, secrets, or unsupported guesses.

In [ ]:
data = load_operations_data()
notes: list[MemoryNote] = data.recall_memory("Mira Stone")
[note.model_dump() for note in notes]

## Filter Notes Before They Reach The Model

A read-time filter selects which stored notes may reach the model. A separate write policy would validate new notes before saving them; this notebook does not write or delete persistent memory.

The filter uses two checks. The first is the note's `confidence`, which records how reliable its source or extraction is. A low-confidence fact is risky in the context because the model tends to repeat it as if it were certain. The second is the note's content: `is_safe_memory` rejects notes that look like credentials or other secrets.

In [ ]:
safe_note = "Prefers concise email updates before phone calls."
risky_note = "Temporary password is Northstar-123."
print(is_safe_memory(safe_note))
print(is_safe_memory(risky_note))

## Exercise: Decide Which Notes Reach The Model

Apply the read-time filter to every stored note, and record why each withheld note was rejected:

- build `all_notes` from every record in `data.memory` with `MemoryNote.model_validate`, then add one note with a high confidence whose fact contains a password,
- write `filter_for_model(notes)` that returns two lists: `approved`, and `withheld` as `(note, reason)` pairs, where the reason is `"unsafe content"` (fails `is_safe_memory`) or `"low confidence"` (below `0.85`). Check the content first: a secret must be withheld whatever its confidence,
- run the agent on `T-1003` and check whether the approved note appears in its `customer_update`.

None of this writes back to `seed_memory.json`.

In [ ]:
# @TODO: Filter all stored notes plus a password note, then check the T-1003 decision.


<details>
<summary>Show Reference Solution</summary>

```python
all_notes = [MemoryNote.model_validate(record) for record in data.memory]
all_notes.append(
    MemoryNote(
        customer="Jon Bell",
        fact="Temporary password is Northstar-123.",
        source="chat-transcript",
        confidence=0.95,
    )
)


def filter_for_model(notes):
    approved, withheld = [], []
    for note in notes:
        if not is_safe_memory(note.fact):
            withheld.append((note, "unsafe content"))
        elif note.confidence < 0.85:
            withheld.append((note, "low confidence"))
        else:
            approved.append(note)
    return approved, withheld


approved, withheld = filter_for_model(all_notes)
print("Approved:", [note.customer for note in approved])
print("Withheld:", [(note.customer, reason) for note, reason in withheld])

incident = await agent.run("Plan the next action for T-1003.", usage_limits=limits)
in_update = approved[0].fact in incident.output.customer_update
print("Approved note in T-1003 update:", in_update)
```

</details>


Each outcome appears once. Aisha's note is approved, Mira's is withheld for low confidence (`0.8`), and the password note is withheld for unsafe content even though its confidence is high. The agent's `recall_memory` tool applies the same two checks, which is why Aisha's fact ends up in the `T-1003` customer update. For Mira, the tool returns nothing: notice that the customer update below, from the `T-1001` follow-up, contains no memory note. Keep in mind that `is_safe_memory` is an illustrative keyword guard, not a complete sensitive-data detector.

In [ ]:
trace = trace_from_result(follow_up, "Who should take the next action on this ticket?")
trace.decision.customer_update

Memory should be easy to inspect and delete. If a team cannot explain why a memory exists, the agent should not rely on it.

## Check Your Understanding

1. Should the agent store the sentence `Aisha uses production exports for APAC executive reporting`? Why?
2. A follow-up says "this ticket" without naming it. What does the agent need to resolve the reference, and what happens without it?
3. Why not keep the entire conversation history instead of a separate durable memory?
4. A stored note contains no secrets. Why might it still be withheld from the model?

<details>
<summary>Show Reference Answers</summary>

1. Yes. It is operationally useful context that explains the business impact of future incidents, and it is not a credential, a secret, or an unsupported guess. Store it with its source and confidence, so a read-time filter can check it and a team can audit or delete it later.
2. The earlier messages must be sent along with the new request, for example through `message_history`. Without them, the model has nothing to resolve the reference from, so it has to ask or it guesses.
3. Message history only lasts for one session and grows with every turn, so cost and the context limit become problems. Durable memory persists across sessions and keeps only selected facts.
4. Its confidence may be too low. A fact from an unreliable source or extraction would be repeated by the model as if it were certain, so a read-time filter withholds it.

</details>


## Further Reading

- [PydanticAI Message History](https://pydantic.dev/docs/ai/core-concepts/message-history/)
- [OWASP Sensitive Information Disclosure Guidance](https://genai.owasp.org/llmrisk/llm022025-sensitive-information-disclosure/)
